# COMP8410 A1 - Week 2 run (Colab)
Before running: set Runtime > Change runtime type > A100 GPU (or L4), add Colab Secrets `HF_TOKEN` (and `GITHUB_TOKEN` because the repo is private), upload the 15 images to `My Drive/comp8410_a1/w2_listing/`.
Keep this tab open and the screen awake. Run the cells in order; check the GPU and compute units per hour before the real run.
Full notes: `colab/setup_cell.md`.

## Cell 1: Drive, code, packages

In [ ]:
%cd /content
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'w2_listing']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"

!pip -q install -r requirements.txt
!pip -q install -U transformers accelerate sentencepiece protobuf
!pip -q install git+https://github.com/huggingface/diffusers

## Cell 2: link Drive folders, log in, check the GPU

In [ ]:
!curl -s -o /dev/null -w "%{http_code}\n" -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev/resolve/main/model_index.json

In [ ]:
from google.colab import userdata
os.environ['HF_HOME'] = '/content/hf_cache'   # local disk, not Drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # inherited by every !python process
!rm -rf results outputs
!ln -s {BASE}/results results
!ln -s {BASE}/outputs outputs
!mkdir -p data/processed && ln -sfn {BASE}/w2_listing data/processed/w2_listing

from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests

## Clear the smoke test results

In [ ]:
!rm -f results/w2.jsonl
!rm -rf outputs/w2

## Cell 3: smoke test, then the real run

In [ ]:
!python pairs/w2_image_editing/run_w2.py --model flux1-kontext-dev
!python pairs/w2_image_editing/run_w2.py --model qwen-image-edit-2509

In [ ]:
!python pairs/w2_image_editing/score_w2.py
!python -m a1eval summarize results/w2.jsonl --out report_assets/w2

In [ ]:
# One cell: results table + charts + image comparison for Week 2
import json, math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

N_IMAGES = 4                       # how many items to show side by side
ROOT = Path('.')
rows = [json.loads(l) for l in open(ROOT / 'results/w2.jsonl')]
df = pd.DataFrame(rows)
meta = pd.json_normalize(df['meta']).add_prefix('m_') if 'meta' in df else pd.DataFrame()
df = pd.concat([df.drop(columns=['meta']), meta], axis=1)
models = [m for m in ['flux1-kontext-dev', 'qwen-image-edit-2509'] if m in set(df.model)] or sorted(set(df.model))
COL = {'flux1-kontext-dev': '#1b6ca8', 'qwen-image-edit-2509': '#d1495b'}
short = {'flux1-kontext-dev': 'FLUX Kontext', 'qwen-image-edit-2509': 'Qwen-Edit'}

# ---- 1. summary table (one row per model)
def get(metric): return df[df.metric == metric].groupby('model').value
summ = pd.DataFrame({
    'items': get('latency_s').count(),
    'latency_s (mean)': get('latency_s').mean().round(1),
    'peak_vram_gb (max)': get('peak_vram_gb').max().round(1),
})
for m in ['edge_ssim', 'clip_i', 'clip_gain']:
    if (df.metric == m).any(): summ[f'{m} (mean)'] = get(m).mean().round(3)
for c in ['m_gpu', 'm_offload']:
    if c in df: summ[c[2:]] = df[df.metric == 'latency_s'].groupby('model')[c].first()
display(summ.loc[[m for m in models if m in summ.index]])

# ---- 2. charts
scored = [m for m in ['edge_ssim', 'clip_i', 'clip_gain'] if (df.metric == m).any()]
fig, axes = plt.subplots(1, 2 + len(scored), figsize=(4.2 * (2 + len(scored)), 3.6), squeeze=False)
axes = axes[0]
def bars(ax, metric, title, ylabel, agg='mean'):
    d = df[df.metric == metric]
    vals = [getattr(d[d.model == m].value, agg)() if (d.model == m).any() else 0 for m in models]
    err = [d[d.model == m].value.std() / math.sqrt(max(1, (d.model == m).sum())) if (d.model == m).sum() > 1 else 0 for m in models]
    ax.bar([short.get(m, m) for m in models], vals, yerr=err, color=[COL.get(m, '#777') for m in models], capsize=4)
    for i, v in enumerate(vals): ax.text(i, v, f'{v:.2f}' if v < 20 else f'{v:.0f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
bars(axes[0], 'latency_s', 'Time per edit', 'seconds')
bars(axes[1], 'peak_vram_gb', 'Peak GPU memory', 'GB', 'max')
labels = {'edge_ssim': ('Layout kept (higher = better)', 'edge SSIM'),
          'clip_i': ('Content kept (higher = better)', 'CLIP-I'),
          'clip_gain': ('Edit achieved (higher = better)', 'CLIP gain')}
for ax, m in zip(axes[2:], scored): bars(ax, m, *labels[m])
plt.tight_layout(); plt.show()
if not scored: print('Quality metrics not computed yet: run score_w2.py, then re-run this cell.')

# ---- 3. image comparison: source | model A | model B (first N items that every model finished)
ids = [i for i in dict.fromkeys(df[df.metric == 'latency_s'].item_id)
       if all((ROOT / f'outputs/w2/{m}/{i}.png').exists() for m in models)][:N_IMAGES]
if ids:
    items = pd.read_csv('pairs/w2_image_editing/items.csv').set_index('item_id')
    fig, axs = plt.subplots(len(ids), 1 + len(models), figsize=(4.4 * (1 + len(models)), 3.4 * len(ids)), squeeze=False)
    for r, i in enumerate(ids):
        paths = [ROOT / items.loc[i, 'source']] + [ROOT / f'outputs/w2/{m}/{i}.png' for m in models]
        names = ['Original'] + [short.get(m, m) for m in models]
        for c, (p, n) in enumerate(zip(paths, names)):
            axs[r][c].imshow(Image.open(p)); axs[r][c].axis('off')
            axs[r][c].set_title(f"{n}\n{i} - {items.loc[i, 'edit_type']}" if c == 0 else n, fontsize=9)
    plt.tight_layout(); plt.show()